# Subtitle Overlay STT - Streaming Server

Levanta el servidor WebSocket de streaming para desarrollo en Colab. El mismo código fuente vive en `scripts/stt_stream_server.py` para poder moverlo después a una VM/VPS GPU.

In [ ]:
!pip install -q faster-whisper fastapi uvicorn[standard] pyngrok websockets


## Traer el repo

Si el repo ya está montado en Colab/Drive, ajustá `REPO_DIR`. Si no, dejá que clone desde GitHub.

In [ ]:
import os

REPO_URL = "https://github.com/Nacholazabal/subtitle_overlay_fw.git"
REPO_BRANCH = "dev/process_all_in_colab"
REPO_DIR = "/content/subtitle_overlay_fw"

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin {REPO_BRANCH}
    !git -C {REPO_DIR} checkout {REPO_BRANCH}
    !git -C {REPO_DIR} pull --ff-only origin {REPO_BRANCH}

os.chdir(REPO_DIR)
print("Repo:", os.getcwd(), "branch:", REPO_BRANCH)

required = [
    "scripts/stt_stream_server.py",
    "scripts/stt_stream_protocol.py",
    "scripts/stt_receiver.py",
]
missing = [path for path in required if not os.path.exists(path)]
if missing:
    raise RuntimeError(
        "Este Colab clono una version del repo sin los archivos streaming. "
        "Primero hay que pushear los cambios locales a GitHub y despues rerunear esta celda. "
        f"Faltan: {missing}"
    )


## Configuración

Defaults iniciales: `small`, ventana máxima 3.0 s, silencio 0.35 s, parciales cada 0.5 s.

In [ ]:
import os

MODEL = "small"
DEVICE = "cuda"
COMPUTE_TYPE = "float16"
PORT = 8765

NGROK_AUTHTOKEN = os.environ.get("NGROK_AUTHTOKEN")
try:
    from google.colab import userdata
    NGROK_AUTHTOKEN = NGROK_AUTHTOKEN or userdata.get("NGROK_AUTHTOKEN")
except Exception:
    pass

if not NGROK_AUTHTOKEN:
    raise RuntimeError(
        "Falta NGROK_AUTHTOKEN. En Colab, agregalo en Secrets con nombre "
        "NGROK_AUTHTOKEN, o setealo con os.environ antes de esta celda."
    )


In [ ]:
from pyngrok import ngrok
import threading
import time
import uvicorn

from scripts.stt_stream_server import ServerConfig, create_app

ngrok.set_auth_token(NGROK_AUTHTOKEN)

config = ServerConfig(
    host="0.0.0.0",
    port=PORT,
    model=MODEL,
    device=DEVICE,
    compute_type=COMPUTE_TYPE,
    max_window_sec=3.0,
    min_silence_sec=0.35,
    partial_sec=0.5,
    partial_agreement=1,
    beam_size=5,
    vad_filter=True,
)

app = create_app(config)

def run_server():
    uvicorn.run(app, host=config.host, port=config.port, log_level="info")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(3)

tunnel = ngrok.connect(PORT, bind_tls=True)
public_url = str(tunnel.public_url)
stream_url = public_url.replace("https://", "wss://") + "/stt/stream"

print("=" * 80)
print("Streaming STT server listo")
print("HTTP health:", public_url + "/health")
print("WebSocket:", stream_url)
print("En WSL:")
print(f"  STT_STREAM_URL={stream_url} ./scripts/run_stt_colab_stream.sh")
print("=" * 80)

while True:
    time.sleep(1)
